# Lab 02 - Muc tieu 4: Moi truong & Bien doi khi hau

**Thanh vien phu trach:** Pham Quang Vinh (23120202)

**Mo ta muc tieu:**
Phan tich xu huong phat thai CO2, muc do su dung nang luong tai tao, va dien tich rung tren toan cau giai doan 2000-2023.
Tim hieu xem nang luong tai tao co dang tang khong? Dien tich rung thay doi ra sao? Quoc gia nao phat thai nhieu nhat?

**Cau hoi phan tich:**
1. Xu huong phat thai CO2 toan cau 2000-2023 nhu the nao, theo tung khu vuc?
2. Co2 per capita phan bo nhu the nao tren ban do the gioi?
3. Nang luong tai tao co tang giua cac khu vuc tu 2000 den 2021 khong?
4. Dien tich rung thay doi ra sao theo quoc gia va theo thoi gian?

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

In [2]:
cwd = Path.cwd()
if (cwd / "WDIEXCEL.xlsx").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "WDIEXCEL.xlsx").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Khong tim thay WDIEXCEL.xlsx o thu muc hien tai hoac thu muc cha.")

SOURCE_FILE = PROJECT_ROOT / "WDIEXCEL.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data_output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source file: {SOURCE_FILE.name}")
print(f"Output dir:  {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Project root: /home/pham-quang-vinh/School/HK2_NAM3/Trực quan hóa dữ liệu/Lab02-Data-Visualization
Source file: WDIEXCEL.xlsx
Output dir:  data_output


## 4.1. Khảo sát indicator môi trường

Bốn indicator được chọn đều thuộc WDI và phù hợp với mục tiêu phân tích về môi trường và biến đổi khí hậu:

| Indicator Code | Tên chỉ số | Ý nghĩa phân tích |
|---|---|---|
| `EN.GHG.CO2.MT.CE.AR5` | CO2 Emissions Total (Mt CO2e) | Tổng lượng phát thải khí nhà kính (trừ LULUCF) — đo quy mô phát thải quốc gia |
| `EN.GHG.CO2.PC.CE.AR5` | CO2 Emissions Per Capita (t CO2e/cap) | Lượng phát thải trung bình mỗi người — so sánh công bằng giữa các nước lớn/nho |
| `EG.FEC.RNEW.ZS` | Renewable Energy Consumption (% of total) | Tỉ lệ năng lượng tái tạo trong tổng tiêu thụ — đo mức độ chuyển đổi năng lượng |
| `AG.LND.FRST.ZS` | Forest Area (% of land area) | Diện tích rừng chiếm bao nhiêu % diện tích đất — theo dõi mất rừng/phục hồi rừng |

**Lưu ý quan trọng:**
- `EN.GHG.CO2.MT.CE.AR5` và `EN.GHG.CO2.PC.CE.AR5` là mã GHG mới (thay thế mã cũ `EN.ATM.CO2E.*` đã bị loại bỏ), đã xác minh có trong WDIEXCEL.xlsx.
- `EG.FEC.RNEW.ZS` chỉ có dữ liệu đến năm 2021 cho hầu hết quốc gia — giai đoạn phân tích cho chỉ số này là 2000-2021.

In [3]:
INDICATORS = {
    "EN.GHG.CO2.MT.CE.AR5": {
        "clean_name": "CO2_Emissions_Total_MtCO2e",
        "analysis_role": "Total greenhouse gas emissions (excl. LULUCF), measures national emission scale",
    },
    "EN.GHG.CO2.PC.CE.AR5": {
        "clean_name": "CO2_Emissions_Per_Capita",
        "analysis_role": "Per capita carbon footprint, enables fair cross-country comparison",
    },
    "EG.FEC.RNEW.ZS": {
        "clean_name": "Renewable_Energy_Pct",
        "analysis_role": "Share of renewable energy in total consumption, measures energy transition progress",
    },
    "AG.LND.FRST.ZS": {
        "clean_name": "Forest_Area_Pct",
        "analysis_role": "Forest coverage as share of land area, tracks deforestation or reforestation",
    },
}

indicator_codes = list(INDICATORS)
years = [str(year) for year in range(2000, 2024)]

series_cols = [
    "Series Code",
    "Topic",
    "Indicator Name",
    "Long definition",
    "Unit of measure",
    "Periodicity",
    "Source",
]
series = pd.read_excel(SOURCE_FILE, sheet_name="Series", usecols=series_cols)
indicator_audit = (
    series[series["Series Code"].isin(indicator_codes)]
    .copy()
    .assign(
        Clean_Column=lambda df: df["Series Code"].map(
            {code: meta["clean_name"] for code, meta in INDICATORS.items()}
        ),
        Analysis_Role=lambda df: df["Series Code"].map(
            {code: meta["analysis_role"] for code, meta in INDICATORS.items()}
        ),
        Available=True,
    )
    .sort_values("Series Code")
)

missing_codes = sorted(set(indicator_codes) - set(indicator_audit["Series Code"]))
if missing_codes:
    raise ValueError(f"Cac indicator khong co trong sheet Series: {missing_codes}")

# indicator_audit.to_csv(OUTPUT_DIR / "wdi_environment_indicator_audit.csv", index=False, encoding="utf-8-sig")
print(f"Tat ca {len(indicator_audit)} indicator deu co trong WDI. Audit file da xuat.")
indicator_audit

Tat ca 4 indicator deu co trong WDI. Audit file da xuat.


,Series Code,Topic,Indicator Name,Long definition,Unit of measure,Periodicity,Source,Clean_Column,Analysis_Role,Available
15,AG.LND.FRST.ZS,Environment: Land use,Forest area (% of land area),Forest area (% of land area) is the share of total land area that is under natural or planted stands of trees of at least 5 meters in si...,% (share) of land area,Annual,"FAOSTAT, Food and Agriculture Organization of the United Nations (FAO), uri: https://www.fao.org/faostat/en/#data/RL, publisher: Food an...",Forest_Area_Pct,"Forest coverage as share of land area, tracks deforestation or reforestation",True
221,EG.FEC.RNEW.ZS,Environment: Energy production & use,Renewable energy consumption (% of total final energy consumption),Renewable energy consumption is the share of renewables energy in total final energy consumption.,% (share) of final energy consumption,Annual,"IEA Energy Statistics Data Browser, International Energy Agency (IEA), uri: https://www.iea.org/data-and-statistics/data-tools/energy-st...",Renewable_Energy_Pct,"Share of renewable energy in total consumption, measures energy transition progress",True
263,EN.GHG.CO2.MT.CE.AR5,Environment: Emissions,Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e),"A measure of annual emissions of carbon dioxide (CO2), one of the six Kyoto greenhouse gases (GHG), from the agriculture, energy, waste,...",Mt CO2eq,Annual,"EDGAR (Emissions Database for Global Atmospheric Research) Community GHG Database, Joint Research Centre (JRC) - European Commission, ur...",CO2_Emissions_Total_MtCO2e,"Total greenhouse gas emissions (excl. LULUCF), measures national emission scale",True
264,EN.GHG.CO2.PC.CE.AR5,Environment: Emissions,Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita),"Total annual emissions of carbon dioxide (CO2), one of the six Kyoto greenhouse gases (GHG), from the agriculture, energy, waste, and in...",t CO2e/capita,Annual,"EDGAR (Emissions Database for Global Atmospheric Research) Community GHG Database, Joint Research Centre (JRC) - European Commission, ur...",CO2_Emissions_Per_Capita,"Per capita carbon footprint, enables fair cross-country comparison",True


In [4]:
country_cols = ["Country Code", "Short Name", "Table Name", "Region", "Income Group"]
country = pd.read_excel(SOURCE_FILE, sheet_name="Country", usecols=country_cols)

country_clean = country.rename(
    columns={
        "Country Code": "Country_Code",
        "Short Name": "Short_Name",
        "Table Name": "Country_Name_Table",
        "Income Group": "Income_Group",
    }
)

# Trong WDI, cac dong aggregate nhu World, income groups, regions thuong co Region bi trong.
real_countries = country_clean[country_clean["Region"].notna()].copy()
aggregate_rows = country_clean[country_clean["Region"].isna()].copy()

print(f"Country metadata rows: {len(country_clean):,}")
print(f"Real countries kept:    {len(real_countries):,}")
print(f"Aggregate rows removed: {len(aggregate_rows):,}")

region_summary = real_countries["Region"].value_counts().rename_axis("Region").reset_index(name="Country_Count")
region_summary

Country metadata rows: 265
Real countries kept:    217
Aggregate rows removed: 48


,Region,Country_Count
0,Europe & Central Asia,58
1,Sub-Saharan Africa,48
2,Latin America & Caribbean,42
3,East Asia & Pacific,37
4,Middle East & North Africa,23
5,South Asia,6
6,North America,3


In [5]:
data_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"] + years
raw = pd.read_excel(SOURCE_FILE, sheet_name="Data", usecols=data_cols)

environment_raw = raw[
    raw["Indicator Code"].isin(indicator_codes)
    & raw["Country Code"].isin(real_countries["Country_Code"])
].copy()

availability_rows = []
for code in indicator_codes:
    sub = environment_raw[environment_raw["Indicator Code"].eq(code)].copy()
    values = sub[years]
    yearly_non_null = values.notna().sum()
    availability_rows.append(
        {
            "Indicator_Code": code,
            "Clean_Column": INDICATORS[code]["clean_name"],
            "Indicator_Name": sub["Indicator Name"].iloc[0] if len(sub) else None,
            "Country_Rows": len(sub),
            "Countries_With_Any_Data": int(values.notna().any(axis=1).sum()),
            "Non_Null_Cells_2000_2023": int(values.notna().sum().sum()),
            "First_Year_With_Data": int(yearly_non_null[yearly_non_null.gt(0)].index.min()),
            "Last_Year_With_Data": int(yearly_non_null[yearly_non_null.gt(0)].index.max()),
        }
    )

availability = pd.DataFrame(availability_rows)
print("=== Kiem tra availability cua 4 indicators (giai doan 2000-2023, chi real countries) ===")
print()
availability

=== Kiem tra availability cua 4 indicators (giai doan 2000-2023, chi real countries) ===



,Indicator_Code,Clean_Column,Indicator_Name,Country_Rows,Countries_With_Any_Data,Non_Null_Cells_2000_2023,First_Year_With_Data,Last_Year_With_Data
0,EN.GHG.CO2.MT.CE.AR5,CO2_Emissions_Total_MtCO2e,Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e),217,203,4872,2000,2023
1,EN.GHG.CO2.PC.CE.AR5,CO2_Emissions_Per_Capita,Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita),217,203,4872,2000,2023
2,EG.FEC.RNEW.ZS,Renewable_Energy_Pct,Renewable energy consumption (% of total final energy consumption),217,212,4702,2000,2022
3,AG.LND.FRST.ZS,Forest_Area_Pct,Forest area (% of land area),217,214,5065,2000,2023


## 4.3. Giải thích ý nghĩa metric và lý do chọn

Để phân tích thực trạng và xu hướng của mục tiêu Môi trường & Biến đổi khí hậu, 4 chỉ số dưới đây đã được lựa chọn từ bộ dữ liệu WDI:

1. **CO2_Emissions_Total_MtCO2e (`EN.GHG.CO2.MT.CE.AR5`)**:
   - *Ý nghĩa:* Tổng lượng phát thải CO2 (triệu tấn CO2 tương đương), không tính đến thay đổi mục đích sử dụng đất và lâm nghiệp (LULUCF).
   - *Lý do chọn:* Đây là thước đo quy mô phát thải tuyệt đối ở cấp độ quốc gia, phản ánh đóng góp tổng thể của từng nước vào hiện tượng nóng lên toàn cầu.

2. **CO2_Emissions_Per_Capita (`EN.GHG.CO2.PC.CE.AR5`)**:
   - *Ý nghĩa:* Lượng phát thải CO2 bình quân trên mỗi người dân (tấn CO2/người).
   - *Lý do chọn:* Chỉ số này cho phép so sánh công bằng hiệu quả giảm phát thải giữa các quốc gia có quy mô dân số khác nhau, đồng thời thể hiện dấu chân carbon cá nhân của các nền kinh tế.

3. **Renewable_Energy_Pct (`EG.FEC.RNEW.ZS`)**:
   - *Ý nghĩa:* Tỷ lệ tiêu thụ năng lượng tái tạo trên tổng năng lượng tiêu thụ cuối cùng (%).
   - *Lý do chọn:* Đo lường tiến trình chuyển dịch cơ cấu năng lượng từ hóa thạch sang năng lượng sạch, là giải pháp cốt lõi để đạt mục tiêu Net Zero.

4. **Forest_Area_Pct (`AG.LND.FRST.ZS`)**:
   - *Ý nghĩa:* Diện tích đất rừng chiếm bao nhiêu phần trăm tổng diện tích đất tự nhiên của quốc gia (%).
   - *Lý do chọn:* Rừng đóng vai trò như các bể hấp thụ carbon tự nhiên (carbon sinks). Chỉ số này phản ánh tác động của quá trình mất rừng (deforestation) hoặc nỗ lực tái trồng rừng (reforestation) của quốc gia.

## 4.2. Quy uoc tien xu ly

Cac buoc tien xu ly (dong nhat voi cac thanh vien khac trong nhom):

1. Chi giu 4 indicators moi truong da xac minh o muc 4.1.
2. Chi giu cac quoc gia that, loai aggregate rows bang metadata `Country.Region`.
3. Gioi han giai doan phan tich `2000-2023`.
4. Voi tung cap `Country_Code` va `Indicator_Code`, chi noi suy tuyen tinh cac khoang missing nam **ben trong chuoi** co do dai toi da 2 nam.
5. Neu mot cap country-indicator co khoang missing ben trong dai hon 2 nam thi loai cap do khoi du lieu sach de tranh tao du lieu gia.
6. Khong extrapolate cho cac nam dau/cuoi nam ngoai pham vi quan sat cua indicator.
7. Xuat du lieu dang wide de nap vao Tableau, moi dong la mot `Country_Code` - `Year`.

**Ghi chu ve `Renewable_Energy_Pct` (`EG.FEC.RNEW.ZS`):**
Indicator nay chi co du lieu den 2021 cho hau het quoc gia. Cac o nam 2022-2023 se bi NaN sau tien xu ly — day la dac diem cua du lieu goc, khong phai loi xu ly.

In [6]:
def max_missing_run(values: pd.Series) -> int:
    max_run = 0
    current = 0
    for value in values:
        if pd.isna(value):
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run


def max_internal_missing_run(values: pd.Series) -> int:
    observed_positions = np.flatnonzero(values.notna().to_numpy())
    if len(observed_positions) == 0:
        return -1
    if len(observed_positions) == 1:
        return 0
    internal = values.iloc[observed_positions[0] : observed_positions[-1] + 1]
    return max_missing_run(internal)


def clean_country_indicator(row: pd.Series) -> tuple[str, pd.Series, int]:
    values = pd.to_numeric(row[years], errors="coerce")
    observed_positions = np.flatnonzero(values.notna().to_numpy())

    if len(observed_positions) == 0:
        return "dropped_no_data", values, 0

    max_gap = max_internal_missing_run(values)
    if max_gap > 2:
        return "dropped_internal_gap_gt_2", values, 0

    cleaned = values.copy()
    first_pos = observed_positions[0]
    last_pos = observed_positions[-1]
    internal = cleaned.iloc[first_pos : last_pos + 1]
    interpolated_internal = internal.interpolate(method="linear", limit=2, limit_area="inside")
    cleaned.iloc[first_pos : last_pos + 1] = interpolated_internal

    imputed_count = int(values.isna().sum() - cleaned.isna().sum())
    return "kept", cleaned, imputed_count

In [7]:
long_records = []
cleaning_records = []

for _, row in environment_raw.iterrows():
    status, cleaned_values, imputed_count = clean_country_indicator(row)
    original_values = pd.to_numeric(row[years], errors="coerce")
    max_internal_gap = max_internal_missing_run(original_values)

    cleaning_records.append(
        {
            "Country_Code": row["Country Code"],
            "Country_Name": row["Country Name"],
            "Indicator_Code": row["Indicator Code"],
            "Indicator_Name": row["Indicator Name"],
            "Clean_Column": INDICATORS[row["Indicator Code"]]["clean_name"],
            "Status": status,
            "Observed_Cells_Before": int(original_values.notna().sum()),
            "Observed_Cells_After": int(cleaned_values.notna().sum()) if status == "kept" else 0,
            "Imputed_Cells": imputed_count if status == "kept" else 0,
            "Max_Internal_Missing_Run": max_internal_gap,
        }
    )

    if status != "kept":
        continue

    value_col = INDICATORS[row["Indicator Code"]]["clean_name"]
    for year, value in cleaned_values.items():
        if pd.notna(value):
            long_records.append(
                {
                    "Country_Name": row["Country Name"],
                    "Country_Code": row["Country Code"],
                    "Year": int(year),
                    "Indicator_Code": row["Indicator Code"],
                    "Indicator_Name": row["Indicator Name"],
                    "Metric": value_col,
                    "Value": float(value),
                }
            )

environment_long = pd.DataFrame(long_records)
cleaning_report = pd.DataFrame(cleaning_records)

# cleaning_report.to_csv(OUTPUT_DIR / "wdi_environment_cleaning_report.csv", index=False, encoding="utf-8-sig")

status_summary = (
    cleaning_report
    .groupby(["Indicator_Code", "Clean_Column", "Status"], as_index=False)
    .agg(
        Country_Indicator_Count=("Country_Code", "count"),
        Observed_Cells_Before=("Observed_Cells_Before", "sum"),
        Observed_Cells_After=("Observed_Cells_After", "sum"),
        Imputed_Cells=("Imputed_Cells", "sum"),
    )
)
print("=== Ket qua tien xu ly theo indicator ===")
status_summary

=== Ket qua tien xu ly theo indicator ===


,Indicator_Code,Clean_Column,Status,Country_Indicator_Count,Observed_Cells_Before,Observed_Cells_After,Imputed_Cells
0,AG.LND.FRST.ZS,Forest_Area_Pct,dropped_no_data,3,0,0,0
1,AG.LND.FRST.ZS,Forest_Area_Pct,kept,214,5065,5065,0
2,EG.FEC.RNEW.ZS,Renewable_Energy_Pct,dropped_no_data,5,0,0,0
3,EG.FEC.RNEW.ZS,Renewable_Energy_Pct,kept,212,4702,4702,0
4,EN.GHG.CO2.MT.CE.AR5,CO2_Emissions_Total_MtCO2e,dropped_no_data,14,0,0,0
5,EN.GHG.CO2.MT.CE.AR5,CO2_Emissions_Total_MtCO2e,kept,203,4872,4872,0
6,EN.GHG.CO2.PC.CE.AR5,CO2_Emissions_Per_Capita,dropped_no_data,14,0,0,0
7,EN.GHG.CO2.PC.CE.AR5,CO2_Emissions_Per_Capita,kept,203,4872,4872,0


In [8]:
environment_wide = (
    environment_long
    .pivot_table(
        index=["Country_Name", "Country_Code", "Year"],
        columns="Metric",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)
environment_wide.columns.name = None

environment_wide = environment_wide.merge(
    real_countries[["Country_Code", "Region", "Income_Group"]],
    on="Country_Code",
    how="left",
)

metric_columns = [INDICATORS[code]["clean_name"] for code in indicator_codes]
for col in metric_columns:
    if col not in environment_wide.columns:
        environment_wide[col] = np.nan

environment_wide = environment_wide[
    ["Country_Name", "Country_Code", "Region", "Income_Group", "Year"] + metric_columns
].sort_values(["Region", "Country_Name", "Year"]).reset_index(drop=True)

environment_wide.to_csv(OUTPUT_DIR / "wdi_environment.csv", index=False, encoding="utf-8-sig")

print(f"Rows exported:     {len(environment_wide):,}")
print(f"Countries exported: {environment_wide['Country_Code'].nunique():,}")
print(f"Years exported:    {environment_wide['Year'].min()}-{environment_wide['Year'].max()}")
print(f"Main CSV:          {OUTPUT_DIR / 'wdi_environment.csv'}")

environment_wide.head(10)

Rows exported:     5,143
Countries exported: 216
Years exported:    2000-2023
Main CSV:          /home/pham-quang-vinh/School/HK2_NAM3/Trực quan hóa dữ liệu/Lab02-Data-Visualization/data_output/wdi_environment.csv


,Country_Name,Country_Code,Region,Income_Group,Year,CO2_Emissions_Total_MtCO2e,CO2_Emissions_Per_Capita,Renewable_Energy_Pct,Forest_Area_Pct
0,American Samoa,ASM,East Asia & Pacific,High income,2000,0.0001,0.001759,0.0,88.65
1,American Samoa,ASM,East Asia & Pacific,High income,2001,0.0001,0.001753,0.0,88.50
2,American Samoa,ASM,East Asia & Pacific,High income,2002,0.0001,0.001752,0.0,88.35
3,American Samoa,ASM,East Asia & Pacific,High income,2003,0.0001,0.001755,0.0,88.20
4,American Samoa,ASM,East Asia & Pacific,High income,2004,0.0001,0.001760,0.0,88.05
5,American Samoa,ASM,East Asia & Pacific,High income,2005,0.0001,0.001766,0.0,87.90
6,American Samoa,ASM,East Asia & Pacific,High income,2006,0.0001,0.001774,0.0,87.75
7,American Samoa,ASM,East Asia & Pacific,High income,2007,0.0001,0.001782,0.0,87.60
8,American Samoa,ASM,East Asia & Pacific,High income,2008,0.0001,0.001791,0.0,87.45
9,American Samoa,ASM,East Asia & Pacific,High income,2009,0.0001,0.001801,0.0,87.30


## 4.7. Thống kê mô tả cơ bản

Dưới đây là thống kê mô tả cho các biến mục tiêu Môi trường & Biến đổi khí hậu sau khi đã tiền xử lý dữ liệu (loại bỏ aggregate rows, lọc quốc gia thực, nội suy các khoảng khuyết ngắn ≤ 2 năm):

In [9]:
final_quality = pd.DataFrame(
    [
        {
            "Column": col,
            "Non_Null_Cells": int(environment_wide[col].notna().sum()),
            "Countries_With_Data": int(environment_wide.loc[environment_wide[col].notna(), "Country_Code"].nunique()),
            "Min_Value": float(environment_wide[col].min(skipna=True)),
            "Max_Value": float(environment_wide[col].max(skipna=True)),
            "Mean_Value": round(float(environment_wide[col].mean(skipna=True)), 3),
            "Median_Value": round(float(environment_wide[col].median(skipna=True)), 3),
            "Std_Value": round(float(environment_wide[col].std(skipna=True)), 3),
        }
        for col in metric_columns
    ]
)

print("=== Thong ke mo ta sau tien xu ly ===")
final_quality

=== Thong ke mo ta sau tien xu ly ===


,Column,Non_Null_Cells,Countries_With_Data,Min_Value,Max_Value,Mean_Value,Median_Value,Std_Value
0,CO2_Emissions_Total_MtCO2e,4872,203,0.0,13021.242000,158.584,9.122,787.547
1,CO2_Emissions_Per_Capita,4872,203,0.0,202.865184,4.878,2.298,8.872
2,Renewable_Energy_Pct,4702,212,0.0,98.300000,29.851,19.100,29.535
3,Forest_Area_Pct,5065,214,0.0,95.577213,32.332,30.929,24.523


## 4.4 và 4.5. Kế hoạch worksheet Tableau và nhận xét cần kiểm tra

Dưới đây là kế hoạch chi tiết dựng các worksheet tương ứng với mục tiêu Môi trường & Biến đổi khí hậu trên Tableau:

In [10]:
worksheet_plan = pd.DataFrame([
    {
        "Worksheet": "ENV_01_CO2_Trend",
        "Loại_biểu_đồ": "Stacked Area chart",
        "Trường_chính": "Year, CO2_Emissions_Total_MtCO2e, Region",
        "Mục_đích": "Theo dõi xu hướng tổng phát thải CO2 toàn cầu phân rã theo khu vực.",
        "Nhận_xét_cần_kiểm_tra": "Khu vực nào phát thải lớn nhất, ảnh hưởng của dịch bệnh COVID-2020 gây sụt giảm tạm thời.",
    },
    {
        "Worksheet": "ENV_02_CO2_Per_Capita_Map",
        "Loại_biểu_đồ": "Choropleth Map",
        "Trường_chính": "Country_Name, CO2_Emissions_Per_Capita, Year",
        "Mục_đích": "Trực quan hóa mức độ phát thải bình quân đầu người của các quốc gia.",
        "Nhận_xét_cần_kiểm_tra": "Sự chênh lệch giữa các nước công nghiệp phát triển (thu nhập cao) và các nước đang phát triển.",
    },
    {
        "Worksheet": "ENV_03_Renewable_Energy_Bar",
        "Loại_biểu_đồ": "Grouped Bar chart",
        "Trường_chính": "Region, Renewable_Energy_Pct, Year (2000 vs 2021)",
        "Mục_đích": "So sánh tỷ lệ tiêu thụ năng lượng tái tạo của các vùng giữa năm 2000 và 2021.",
        "Nhận_xét_cần_kiểm_tra": "Khu vực nào có sự chuyển dịch xanh mạnh mẽ nhất, khu vực nào còn chậm trễ.",
    },
    {
        "Worksheet": "ENV_04_Forest_Area_Heatmap",
        "Loại_biểu_đồ": "Heatmap",
        "Trường_chính": "Country_Name, Year, Forest_Area_Pct",
        "Mục_đích": "Theo dõi sự thay đổi diện tích rừng của Top 20 quốc gia có biến động lớn nhất.",
        "Nhận_xét_cần_kiểm_tra": "Các quốc gia mất rừng nghiêm trọng và các quốc gia có sự phục hồi diện tích phủ xanh.",
    }
])
worksheet_plan

,Worksheet,Loại_biểu_đồ,Trường_chính,Mục_đích,Nhận_xét_cần_kiểm_tra
0,ENV_01_CO2_Trend,Stacked Area chart,"Year, CO2_Emissions_Total_MtCO2e, Region",Theo dõi xu hướng tổng phát thải CO2 toàn cầu phân rã theo khu vực.,"Khu vực nào phát thải lớn nhất, ảnh hưởng của dịch bệnh COVID-2020 gây sụt giảm tạm thời."
1,ENV_02_CO2_Per_Capita_Map,Choropleth Map,"Country_Name, CO2_Emissions_Per_Capita, Year",Trực quan hóa mức độ phát thải bình quân đầu người của các quốc gia.,Sự chênh lệch giữa các nước công nghiệp phát triển (thu nhập cao) và các nước đang phát triển.
2,ENV_03_Renewable_Energy_Bar,Grouped Bar chart,"Region, Renewable_Energy_Pct, Year (2000 vs 2021)",So sánh tỷ lệ tiêu thụ năng lượng tái tạo của các vùng giữa năm 2000 và 2021.,"Khu vực nào có sự chuyển dịch xanh mạnh mẽ nhất, khu vực nào còn chậm trễ."
3,ENV_04_Forest_Area_Heatmap,Heatmap,"Country_Name, Year, Forest_Area_Pct",Theo dõi sự thay đổi diện tích rừng của Top 20 quốc gia có biến động lớn nhất.,Các quốc gia mất rừng nghiêm trọng và các quốc gia có sự phục hồi diện tích phủ xanh.


## Kết luận cho công việc của Quang Vinh

- **4.1:** Đã khảo sát và xác nhận 4/4 indicator môi trường trong WDI.
- **4.2:** Đã tiền xử lý dữ liệu môi trường, loại aggregate rows, xử lý missing values thận trọng và xuất ra file dữ liệu sạch `data_output/wdi_environment.csv`.
- **4.3:** Đã giải thích ý nghĩa từng metric và lý do chọn trong notebook.
- **4.4:** Đã đề xuất 4 worksheet Tableau phù hợp: stacked area chart, choropleth map, grouped bar chart và heatmap.
- **4.5:** Đã chuẩn bị các hướng nhận xét cần kiểm tra khi dựng biểu đồ trên Tableau.
- **4.7:** Đã tính thống kê mô tả cơ bản cho các indicator môi trường.

**Lưu ý đặc biệt:** Cột `Renewable_Energy_Pct` (EG.FEC.RNEW.ZS) chỉ có dữ liệu đến 2021 cho hầu hết quốc gia. Các ô năm 2022-2023 là NaN — đây là đặc điểm dữ liệu gốc, không phải lỗi xử lý. Sẽ ghi rõ trong báo cáo.